In [1]:
import pandas as pd
import os

def process_excel_advanced(file_path):
    # 1. 检查文件是否存在
    if not os.path.exists(file_path):
        print(f"错误：找不到文件 - {file_path}")
        return

    try:
        print("正在读取 Excel 文件，请稍候...")
        
        # 2. 读取 Excel
        # dtype={'用户编号': str} 防止长数字变成科学计数法
        df = pd.read_excel(file_path, dtype={'用户编号': str})
        
        if '地市' not in df.columns:
            print("错误：表中未找到“地市”这一列。")
            return

        # 3. 构建输出路径
        # 获取源文件所在目录
        source_dir = os.path.dirname(file_path)
        # 新建文件夹名称
        new_folder_name = "分地市结果"
        # 完整的新文件夹路径
        output_dir = os.path.join(source_dir, new_folder_name)
        
        # 如果文件夹不存在，则创建
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"已创建新文件夹: {output_dir}")

        # 构建输出文件名
        base_name = os.path.basename(file_path)
        file_name_without_ext = os.path.splitext(base_name)[0]
        output_path = os.path.join(output_dir, f"{file_name_without_ext}_处理结果.xlsx")

        print(f"正在处理数据，共 {len(df)} 行...")

        # 定义需要新增的列名列表
        new_columns = [
            "预计完成时间\n（如2026/2/10）",
            "预计送（停）电日期",
            "业扩变更前2月份日均电量\n（比如2月10号完成业扩变更，此处填2月1-10号的日均电量）",
            "业扩变更后2月份日均电量\n（比如2月10号完成业扩变更，此处填2月11-28号的日均电量）",
            "业扩变更后3月份日均电量"
        ]

        # 4. 写入 Excel
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            
            # --- 处理汇总 Sheet ---
            df_summary = df.copy()
            # 插入序号
            df_summary.insert(0, '序号', range(1, len(df_summary) + 1))
            # 添加新列（赋值为空字符串）
            for col in new_columns:
                df_summary[col] = ""
            
            # 写入汇总 Sheet
            df_summary.to_excel(writer, sheet_name='汇总', index=False)
            print(f"  - 已生成 Sheet: 汇总 (共 {len(df_summary)} 条)")

            # --- 处理分地市 Sheet ---
            cities = df['地市'].dropna().unique()
            
            for city in cities:
                # 筛选当前地市的数据
                city_df = df[df['地市'] == city].copy()
                
                # 插入序号（重新编号）
                city_df.insert(0, '序号', range(1, len(city_df) + 1))
                
                # 添加新列
                for col in new_columns:
                    city_df[col] = ""
                
                # 处理 sheet 名称
                sheet_name = str(city)[:31]
                
                # 写入 sheet
                city_df.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"  - 已生成 Sheet: {sheet_name} (共 {len(city_df)} 条)")

        print("-" * 30)
        print(f"处理完成！\n文件已保存至: {output_path}")

    except Exception as e:
        print(f"发生错误: {e}")

if __name__ == "__main__":
    # 你的文件路径
    input_file = r"E:\A智网\业扩分析\26年2月分析\业扩下发地市清单_在途.xlsx"
    
    process_excel_advanced(input_file)

正在读取 Excel 文件，请稍候...
已创建新文件夹: E:\A智网\业扩分析\26年2月分析\分地市结果
正在处理数据，共 1794 行...
  - 已生成 Sheet: 汇总 (共 1794 条)
  - 已生成 Sheet: 武汉供电公司 (共 228 条)
  - 已生成 Sheet: 黄石供电公司 (共 99 条)
  - 已生成 Sheet: 十堰供电公司 (共 88 条)
  - 已生成 Sheet: 襄阳供电公司 (共 161 条)
  - 已生成 Sheet: 黄冈供电公司 (共 193 条)
  - 已生成 Sheet: 鄂州供电公司 (共 52 条)
  - 已生成 Sheet: 荆州供电公司 (共 275 条)
  - 已生成 Sheet: 孝感供电公司 (共 161 条)
  - 已生成 Sheet: 宜昌供电公司 (共 134 条)
  - 已生成 Sheet: 咸宁供电公司 (共 166 条)
  - 已生成 Sheet: 荆门供电公司 (共 92 条)
  - 已生成 Sheet: 随州供电公司 (共 45 条)
  - 已生成 Sheet: 恩施供电公司 (共 98 条)
  - 已生成 Sheet: 神农架供电公司 (共 2 条)
------------------------------
处理完成！
文件已保存至: E:\A智网\业扩分析\26年2月分析\分地市结果\业扩下发地市清单_在途_处理结果.xlsx
